# 02 · Feature Engineering
## Graph-Aware Feature Construction for Fraud Detection

### Prerequisites
Run `01_EDA.ipynb` first. The following must be in memory:
- `df`: merged DataFrame with class labels
- `G`: NetworkX directed graph
- `edges`: edge list DataFrame

---

### What this notebook does
Raw node features ignore the most important signal in a transaction network — **who you transact with matters as much as what you do**. This notebook engineers three new feature families on top of Elliptic's original 166-dimensional vectors:

| Family | Description | Motivation |
|--------|-------------|------------|
| **Structural** | Centrality, degree, clustering | Network position reveals influence and risk |
| **Temporal** | Burst activity, illicit rate at same step, recency | Illicit nodes cluster in time bursts |
| **Neighbourhood** | Illicit neighbour ratio, risk contagion | Guilt-by-association is empirically validated |

### Output
- `full_features.csv`: 181-feature matrix for all 203,769 nodes (required by notebook 03)

In [ ]:
# Imports — all libraries needed for feature engineering
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✅ Starting Feature Engineering')
print(f'Working with {len(df):,} nodes and {G.number_of_edges():,} edges')

## A. Structural Features
These capture each node's position and influence in the network.
Standard ML models that ignore graph structure completely miss these signals.

In [ ]:
# Compute structural centrality features for every node
print('Computing in and out degree...')
in_deg  = dict(G.in_degree())
out_deg = dict(G.out_degree())

# Betweenness centrality — how often does this node sit on the shortest path between others?
# k=500 means we sample 500 pivot nodes — good balance of speed vs accuracy on 200k nodes
print('Computing betweenness centrality (k=500 approximation)...')
betweenness = nx.betweenness_centrality(G, normalized=True, k=500)

# PageRank — how many important nodes point to you?
print('Computing PageRank...')
pagerank = nx.pagerank(G, alpha=0.85, max_iter=300)

# HITS — hub score (points to authorities) and authority score (pointed to by hubs)
print('Computing HITS hub and authority scores...')
hubs, authorities = nx.hits(G, max_iter=100, normalized=True)

node_ids  = df['txId'].tolist()
struct_df = pd.DataFrame({
    'txId'        : node_ids,
    'in_degree'   : [in_deg.get(n, 0)                        for n in node_ids],
    'out_degree'  : [out_deg.get(n, 0)                       for n in node_ids],
    'total_degree': [in_deg.get(n,0) + out_deg.get(n,0)      for n in node_ids],
    'degree_ratio': [out_deg.get(n,0)/(in_deg.get(n,0)+1e-6) for n in node_ids],
    'betweenness' : [betweenness.get(n, 0)                   for n in node_ids],
    'pagerank'    : [pagerank.get(n, 0)                      for n in node_ids],
    'hub_score'   : [hubs.get(n, 0)                          for n in node_ids],
    'auth_score'  : [authorities.get(n, 0)                   for n in node_ids],
})

print(f'✅ Structural features: {struct_df.shape[1]-1} columns')
print(struct_df.describe().round(4))

## B. Temporal Features
These capture when each transaction happened and what was happening around it.
Since illicit activity clusters in burst windows, the conditions at each time step carry signal.

In [ ]:
# Compute temporal context features for every node
print('Computing temporal features...')

ts_df     = df[['txId', 'time_step']].copy()
ts_volume = df.groupby('time_step').size().rename('ts_volume')
ts_illicit = (df[df['class'] == 1]
              .groupby('time_step').size().rename('ts_illicit_count'))

ts_stats = (
    pd.DataFrame({'time_step': range(1, 50)})
    .merge(ts_volume.reset_index(),  on='time_step', how='left')
    .merge(ts_illicit.reset_index(), on='time_step', how='left')
    .fillna(0)
)
# What fraction of labelled nodes at each time step are illicit?
ts_stats['ts_illicit_rate'] = (
    ts_stats['ts_illicit_count'] / (ts_stats['ts_volume'] + 1e-6)
)

ts_df = ts_df.merge(
    ts_stats[['time_step', 'ts_volume', 'ts_illicit_rate']],
    on='time_step', how='left'
)
# Recency — 0 = earliest time step, 1 = latest
ts_df['time_recency'] = (ts_df['time_step'] - 1) / 48.0

print(f'✅ Temporal features: {ts_df.shape[1]-2} columns')
print(ts_df.head(3))

## C. Neighbourhood Features
These capture guilt-by-association — the most powerful signal that standard ML completely ignores.
If 8 out of 10 nodes sending money to you are illicit, that is a strong fraud signal even if your own features look clean.

In [ ]:
# Compute neighbourhood contagion features for every node
# This is the guilt-by-association signal
print('Computing neighbourhood features...')
print('This may take 3-5 minutes on 200k nodes...')

label_map = df.set_index('txId')['class'].to_dict()
records   = []

try:
    for i, node in enumerate(node_ids):
        preds  = list(G.predecessors(node)) if node in G else []
        succs  = list(G.successors(node))   if node in G else []
        all_nb = preds + succs

        nb_lbls  = [label_map.get(n, 0) for n in all_nb]
        n_total  = len(all_nb) + 1e-6

        pred_lbls      = [label_map.get(n, 0) for n in preds]
        pred_ill_ratio = sum(l == 1 for l in pred_lbls) / (len(preds) + 1e-6)

        records.append({
            'txId'              : node,
            'nb_illicit_ratio'  : sum(l == 1 for l in nb_lbls) / n_total,
            'nb_licit_ratio'    : sum(l == 2 for l in nb_lbls) / n_total,
            'nb_unknown_ratio'  : sum(l == 0 for l in nb_lbls) / n_total,
            'nb_total'          : len(all_nb),
            'pred_illicit_ratio': pred_ill_ratio,
        })

        if (i + 1) % 50000 == 0:
            print(f'  Processed {i+1:,} / {len(node_ids):,} nodes...')

    neigh_df = pd.DataFrame(records)
    print(f'✅ Neighbourhood features: {neigh_df.shape[1]-1} columns')

except Exception as e:
    print(f'❌ Error: {e}')

## D. Merge All Features and Save

In [ ]:
# Merge all three feature families with the original Elliptic features
orig_feats = [f'f{i}' for i in range(1, 166)]
base_cols  = ['txId', 'time_step', 'class', 'class_label'] + orig_feats

full_df = (
    df[base_cols]
    .merge(struct_df,                       on='txId', how='left')
    .merge(ts_df.drop('time_step', axis=1), on='txId', how='left')
    .merge(neigh_df,                        on='txId', how='left')
    .fillna(0)
)

print(f'Final feature matrix: {full_df.shape[0]:,} rows x {full_df.shape[1]} columns')
print(f'  Original Elliptic features : 165')
print(f'  Structural features        : {len(struct_df.columns)-1}')
print(f'  Temporal features          : {len(ts_df.columns)-2}')
print(f'  Neighbourhood features     : {len(neigh_df.columns)-1}')
print(f'  Total feature columns      : {full_df.shape[1]-4}')

full_df.to_csv('full_features.csv', index=False)
print('\n✅ Saved → full_features.csv')
print('Next: Run 03_GNN_Classifier.ipynb')